In [1]:
import numpy as np
import time
from gridcp.new_api.detector import GridDetector, DetectorState
from gridcp.new_api.scores import MeanCUSUM
from gridcp.new_api.typing import ArrayLike
import gridcp

In [2]:
def run_online_grid_detector(
    data: ArrayLike,
    detector: GridDetector,
    reset_on_alarm: bool = False,
) -> tuple[DetectorState, dict]:
    """Run a configured GridDetector over a dataset sequentially.

    Parameters
    ----------
    data : ArrayLike
        Sequence of observations. Each element is passed as `x` to
        `detector.update`. For univariate data this is a 1D array of scalars;
        for multivariate data it should be a 2D array of shape (n_samples, n_features).
    detector : GridDetector
        A fully configured detector instance (with `score` and `threshold`
        set). State is initialised internally via `detector.init_state()`.
    reset_on_alarm : bool, optional
        If True, the detector state is reset to a fresh initial state immediately
        after an alarm is raised. This allows the detector to restart tracking
        from the next observation after each detected changepoint.
        Default is False.

    Returns
    -------
    state : DetectorState
        State after each observation, including the initial state at index 0.
        When `reset_on_alarm=False`, `states[i]` is the state after processing
        `data[i - 1]`, so `states[0]` is the fresh initial state.
        When `reset_on_alarm=True`, the index correspondence is broken at each
        alarm: the state is reset before the next observation is processed.
    outputs : list[dict]
        One entry per observation. Each dict contains:
          - ``"index"``: time index (n_samples) when the alarm was raised.
          - ``"alarm"``: bool, True when ``max_score > threshold``.
          - ``"max_score"``: highest penalised score among active candidates.
          - ``"max_score_index"``: grid position of the highest-scoring candidate.
    """
    data = np.asarray(data)
    state = detector.init_state()
    output = None
    for x in data:
        state, output = detector.update(state, x)

    return state, output

In [3]:
def demo(n_samples=100, n_features=1, N=100, reset_on_alarm=False):
    """Simulate toy univariate data with a known mean shift and run the grid detector."""
    rng = np.random.default_rng(seed=42)
    finalmaxx = []
    for i in range(N):
        n_pre, n_post = n_samples // 2, n_samples // 2 + n_samples % 2
        data = np.concatenate(
            [
                rng.normal(loc=0.0, scale=1.0, size=(n_pre, n_features)),
                rng.normal(loc=0.0, scale=1.0, size=(n_post, n_features)),
            ]
        )

        score = MeanCUSUM(n_features)
        detector = GridDetector(score=score, threshold=1000.0)
        state, output = run_online_grid_detector(data, detector, reset_on_alarm)
        finalmaxx.append(output["max_score"])

In [4]:
n_samples = 1000
N = 500

In [5]:
start = time.perf_counter()
demo(n_samples=n_samples, N=N, reset_on_alarm=False)
stop = time.perf_counter()
print(f"Execution time: {stop - start:.4f} seconds")

Execution time: 6.1453 seconds


In [6]:
detector = gridcp.make_univariate_mean_change_detector()
detector.calibrate_false_alarm(alpha=0.05, N=n_samples, K=N, null_dist=np.random.normal)

In [7]:
def benchmark(n_samples=1000, n_features=1, n_repeats=5, top_n=20):
    """Profile and time run_online_grid_detector to identify bottlenecks.

    Runs a Numba warm-up pass first (compilation overhead excluded from timing),
    then times `n_repeats` runs and profiles one representative run with cProfile.

    Parameters
    ----------
    n_samples : int
        Number of observations per run.
    n_features : int
        Observation dimensionality.
    n_repeats : int
        Number of timed repetitions (warm-up excluded).
    top_n : int
        Number of hottest functions to print from cProfile output.
    """
    import cProfile
    import pstats
    import io
    import timeit

    rng = np.random.default_rng(seed=0)
    n_pre = n_samples // 2
    n_post = n_samples - n_pre
    data = np.concatenate(
        [
            rng.normal(0.0, 1.0, size=(n_pre, n_features)),
            rng.normal(5.0, 1.0, size=(n_post, n_features)),
        ]
    )

    detector = GridDetector(score=MeanCUSUM(n_features), threshold=10.0)

    # Warm up Numba JIT compilation — excluded from timing.
    print("Warming up Numba (first call compiles)...")
    run_online_grid_detector(data[: min(50, n_samples)], detector)
    print("Warm-up done.\n")

    # Timing
    def _run():
        run_online_grid_detector(data, detector)

    times = timeit.repeat(_run, number=1, repeat=n_repeats)
    print(
        f"Timing over {n_repeats} runs (n_samples={n_samples}, n_features={n_features}):"
    )
    print(
        f"  min={min(times) * 1000:.2f}ms  mean={sum(times) / len(times) * 1000:.2f}ms  max={max(times) * 1000:.2f}ms\n"
    )

    # cProfile of a single run
    pr = cProfile.Profile()
    pr.enable()
    run_online_grid_detector(data, detector)
    pr.disable()

    buf = io.StringIO()
    ps = pstats.Stats(pr, stream=buf).sort_stats("cumulative")
    ps.print_stats(top_n)
    print(f"cProfile output (top {top_n} by cumulative time):")
    print(buf.getvalue())

In [8]:
benchmark(n_samples=100000, n_features=1, n_repeats=5, top_n=20)

Warming up Numba (first call compiles)...
Warm-up done.

Timing over 5 runs (n_samples=100000, n_features=1):
  min=1610.02ms  mean=1671.75ms  max=1814.57ms

cProfile output (top 20 by cumulative time):
         5772760 function calls (5772755 primitive calls) in 2.434 seconds

   Ordered by: cumulative time
   List reduced from 156 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      2/1    0.048    0.024    2.429    2.429 /var/folders/fs/mnbtdnd93p19btpzynvjn__c0000gn/T/ipykernel_82875/765582170.py:1(run_online_grid_detector)
   100000    0.194    0.000    2.400    0.000 /Users/peraugustmoen/Library/CloudStorage/OneDrive-UniversitetetiOslo/Project6_softwarepaper/G-CHAD/gridcp/new_api/detector.py:99(update)
    99999    0.478    0.000    1.760    0.000 /Users/peraugustmoen/Library/CloudStorage/OneDrive-UniversitetetiOslo/Project6_softwarepaper/G-CHAD/gridcp/new_api/scores/_mean_cusum.py:132(compute_penalised_scores)
    99999    

In [10]:
# ============================================================================
# COMPARISON: Old API vs New API
# ============================================================================

# Generate test data
np.random.seed(42)
n_samples = 500
n_pre = n_samples // 2
data = np.concatenate(
    [
        np.random.normal(loc=0.0, scale=1.0, size=n_pre),
        np.random.normal(loc=3.0, scale=1.0, size=n_samples - n_pre),
    ]
)

print("=" * 70)
print("COMPARING OLD API vs NEW API")
print("=" * 70)
print(f"Data: {n_samples} samples, mean shift at index {n_pre}\n")

# OLD API
print("OLD API (OnlineChangepointDetector):")
print("-" * 70)
old_detector = gridcp.make_univariate_mean_change_detector(penalty_constant=1.0)
for x in data:
    old_detector.update(x)

print(f"  t (samples processed):      {old_detector.t}")
print(f"  max_statistic (maxx):       {old_detector.max_statistic:.6f}")
print(f"  maxpos (best change loc):   {old_detector.maxpos}")
print(f"  alarm:                      {old_detector.alarm}")
print(f"  grid size:                  {len(old_detector._state['grid_list'])}")
print(f"  grid (last 5 points):       {list(old_detector._state['grid_list'])[-5:]}")
print()

# NEW API
print("NEW API (GridDetector):")
print("-" * 70)
new_score = MeanCUSUM(n_features=1)
new_detector = GridDetector(score=new_score, threshold=1.0)
new_state, new_output = run_online_grid_detector(data, new_detector)

print(f"  n_samples:                  {new_state.n_samples}")
print(f"  max_score:                  {new_output.get('max_score', 'N/A'):.6f}")
print(f"  max_score_index (grid pos): {new_output.get('max_score_index', 'N/A')}")
print(f"  alarm:                      {new_output.get('alarm', 'N/A')}")
print(f"  grid size:                  {len(new_state.grid)}")
print(f"  grid (last 5 points):       {new_state.grid[-5:]}")
print()

# COMPARISON TABLE
print("=" * 70)
print("SIDE-BY-SIDE COMPARISON")
print("=" * 70)
print(f"{'Metric':<35} {'OLD API':<20} {'NEW API':<20}")
print("-" * 70)
print(f"{'Total samples processed':<35} {old_detector.t:<20} {new_state.n_samples:<20}")
print(
    f"{'Max statistic':<35} {old_detector.max_statistic:<20.6f} {new_output.get('max_score', 0):<20.6f}"
)
print(
    f"{'Changepoint location':<35} {old_detector.maxpos:<20} {new_output.get('max_score_index', 0):<20}"
)
print(
    f"{'Alarm triggered':<35} {str(old_detector.alarm):<20} {str(new_output.get('alarm', False)):<20}"
)
print(
    f"{'Number of grid candidates':<35} {len(old_detector._state['grid_list']):<20} {len(new_state.grid):<20}"
)
print(
    f"{'Grid points match':<35} {str(len(old_detector._state['grid_list']) == len(new_state.grid)):<20} {'---':<20}"
)
print()

# DIFFERENCES
print("=" * 70)
print("ANALYSIS")
print("=" * 70)
if old_detector.max_statistic > 0 and new_output.get("max_score", 0) > 0:
    abs_diff = abs(old_detector.max_statistic - new_output.get("max_score", 0))
    rel_diff = abs_diff / old_detector.max_statistic * 100
    print(f"Max statistic difference:  {abs_diff:.6f} ({rel_diff:.2f}%)")
else:
    print("Max statistic difference:  Cannot compare (one or both are zero)")

if old_detector.maxpos == new_output.get("max_score_index", 0):
    print(f"Changepoint location:      MATCH ✓")
else:
    print(
        f"Changepoint location:      MISMATCH (old: {old_detector.maxpos}, new: {new_output.get('max_score_index', 'N/A')})"
    )

if old_detector.alarm == new_output.get("alarm", False):
    print(f"Alarm state:               MATCH ✓")
else:
    print(
        f"Alarm state:               MISMATCH (old: {old_detector.alarm}, new: {new_output.get('alarm', False)})"
    )


print("grids:")
print(new_state.grid)
print([-g + 1 for g in old_detector._state["grid_list"]])

COMPARING OLD API vs NEW API
Data: 500 samples, mean shift at index 250

OLD API (OnlineChangepointDetector):
----------------------------------------------------------------------
  t (samples processed):      500
  max_statistic (maxx):       122.575725
  maxpos (best change loc):   241
  alarm:                      True
  grid size:                  16
  grid (last 5 points):       [-494, -496, -498, -499, -500]

NEW API (GridDetector):
----------------------------------------------------------------------
  n_samples:                  500
  max_score:                  122.239015
  max_score_index (grid pos): 257
  alarm:                      True
  grid size:                  16
  grid (last 5 points):       [493, 495, 497, 498, 499]

SIDE-BY-SIDE COMPARISON
Metric                              OLD API              NEW API             
----------------------------------------------------------------------
Total samples processed             500                  500                 
